In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [3]:
# Load  API Keys
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

NameError: name 'load_dotenv' is not defined

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
MODEL_CONFIGS = {
    "openai": {
        "provider": "openai",
        "model": "gpt-5-nano",
    },
    "anthropic": {
        "provider": "anthropic",
        "model": "claude-sonnet-4-5",
        "max_tokens": 1000,
    },
    "gemini": {
        "provider": "openai-compatible",
        "model": "gemini-2.5-flash",
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key": google_api_key,
    },
    "deepseek": {
        "provider": "openai-compatible",
        "model": "deepseek-chat",
        "base_url": "https://api.deepseek.com/v1",
        "api_key": deepseek_api_key,
    },
    "groq": {
        "provider": "openai-compatible",
        "model": "openai/gpt-oss-120b",
        "base_url": "https://api.groq.com/openai/v1",
        "api_key": groq_api_key,
    },
    "ollama": {
        "provider": "openai-compatible",
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",
    },
}


def _anthropic_messages(prompt):
    return [{"role": "user", "content": prompt}]


def generate_answer(prompt, model_cfg):
    provider = model_cfg["provider"]

    if provider == "anthropic":
        client = Anthropic(api_key=anthropic_api_key)
        response = client.messages.create(
            model=model_cfg["model"],
            messages=_anthropic_messages(prompt),
            max_tokens=model_cfg.get("max_tokens", 1000),
        )
        return response.content[0].text

    client = OpenAI(
        api_key=model_cfg.get("api_key"),
        base_url=model_cfg.get("base_url"),
    )
    response = client.chat.completions.create(
        model=model_cfg["model"],
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def run_all_models(prompt, model_configs):
    results = {}
    with ThreadPoolExecutor(max_workers=len(model_configs)) as executor:
        futures = {
            name: executor.submit(generate_answer, prompt, cfg)
            for name, cfg in model_configs.items()
        }
        for name, future in futures.items():
            try:
                results[name] = future.result()
            except Exception as e:
                results[name] = f"ERROR: {e}"

    return results


results = run_all_models(question, MODEL_CONFIGS)

for name, cfg in MODEL_CONFIGS.items():
    model_name = cfg["model"]
    answer = results[name]
    display(Markdown(f"### {model_name}\n{answer}"))
    competitors.append(model_name)
    answers.append(answer)

In [ ]:
# Replaced by the config + parallel execution cell above.

In [ ]:
# Replaced by the config + parallel execution cell above.

In [ ]:
# Replaced by the config + parallel execution cell above.

In [ ]:
# Replaced by the config + parallel execution cell above.

In [ ]:
# Replaced by the config + parallel execution cell above.

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank (1st is the best) and score them 1-100  in order of best to worst.
Give a score 1-100 to the quality of the answer, eg 100 is the perfect quality, it cannot be inproved, while 0 is totally unacceptable.
Respond with JSON, and only JSON, with the following format:
{{"results": [{{"rank":"1","competitor_number":3,"score":98}}, {{"rank":"2","competitor_number":2,"score":70}}, {{"rank":"3","competitor_number":1,"score":22}}, ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [79]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks_and_scores = results_dict["results"]
for index, result in enumerate(ranks_and_scores):
    print(f"{result.get("rank")}: {competitors[result.get("competitor_number")-1]} with score: {result.get("score")}")


1: gpt-5-nano with score: 98
2: gemini-2.5-flash with score: 94
3: deepseek-chat with score: 85
4: claude-sonnet-4-5 with score: 65
5: llama3.2 with score: 20


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>